# 2013–2014 SOXX 原始披露来源

八份原文与来源边界见 `plan.json`。先收集和核对身份，再决定数值抽取。


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex,time
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_2013_2014_reported_concentration_source_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['trust_candidate_index_sha256']==sha(root/'research/stage1_soxx_legacy_basket_history_20261023/candidate_trust_filings.json')
assert plan['pre_2015_series_calendar_audit_sha256']==sha(root/'research/stage1_soxx_legacy_reported_concentration_source_20261023/series_calendar_transition_audit.json')
rawdir=root/'data/snapshots/stage1_soxx_2013_2014_reported_concentration_source_20261023';rawdir.mkdir(parents=True,exist_ok=True)
claim=card/'attempt_started.json';out=card/'transport_result.json'
assert not claim.exists() and not out.exists() and not any(rawdir.iterdir())
claim.write_text(json.dumps({'plan_sha256':sha(card/'plan.json'),'started_at_utc':datetime.now(timezone.utc).isoformat(),'maximum_GETs':8},indent=2)+'\n')
requests=[]
for j,item in enumerate(plan['reports']):
 if j:time.sleep(plan['request']['minimum_seconds_between_requests'])
 raw=rawdir/(item['accession']+'_'+item['primary_document']);headers=rawdir/(raw.name+'.headers.txt')
 started=datetime.now(timezone.utc).isoformat()
 cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
  '--max-time',str(plan['request']['total_timeout_seconds_each']),'--max-filesize',str(plan['request']['max_response_bytes_each']),
  '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
  '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: text/html',item['url']]
 proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds_each']+15)
 ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
 entry={'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'accepted_at_utc':item['accepted_at_utc'],
  'url':item['url'],'requested_at_utc':started,'received_at_utc':ended,'http_status':status,
  'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:400],
  'raw_path':str(raw.relative_to(root)),'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
  'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0}
 requests.append(entry)
 if proc.returncode or status!=200 or entry['final_url']!=item['url'] or not raw.exists() or raw.stat().st_size>plan['request']['max_response_bytes_each']:
  entry['stop_reason']='transport_or_identity_failed';break
receipt={'status':'eight_original_2013_2014_SEC_bodies_received_identity_pending' if len(requests)==8 and all('stop_reason' not in x for x in requests) else 'original_2013_2014_body_batch_stopped',
 'plan_sha256':sha(card/'plan.json'),'GETs_consumed':len(requests),'requests':requests,'SOXX_schedule_identities_qualified':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='soxx-2013-14-original-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in [claim,out,*sorted(rawdir.iterdir())]:
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='soxx-2013-14-original-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out_file:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out_file,stderr=subprocess.PIPE,timeout=120)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   for entry in requests:
    if entry['raw_sha256'] is None:continue
    member=stored.extractfile(Path(entry['raw_path']).name)
    assert member and hashlib.sha256(member.read()).hexdigest()==entry['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=True
 out.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'GETs':len(requests),'raw_MB':round(sum(x['raw_bytes'] for x in requests)/1e6,2),'NAS_restore':True})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'eight_original_2013_2014_SEC_bodies_received_identity_pending', 'GETs': 8, 'raw_MB': 216.48, 'NAS_restore': True}


In [1]:
import json,hashlib,re,html
from pathlib import Path
from datetime import datetime
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_2013_2014_reported_concentration_source_20261023';plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
transport=json.loads((card/'transport_result.json').read_text())
assert transport['status']=='eight_original_2013_2014_SEC_bodies_received_identity_pending' and transport['nas_scoped_restore_verified']
assert transport['plan_sha256']==sha(card/'plan.json') and len(transport['requests'])==8
assert not (card/'identity_result.json').exists()
target='PHLX SOX SEMICONDUCTOR SECTOR INDEX FUND';checks=[]
for item in transport['requests']:
 body=(root/item['raw_path']).read_bytes();assert hashlib.sha256(body).hexdigest()==item['raw_sha256']
 expected=datetime.strptime(item['report_date'],'%Y-%m-%d').strftime('%B %d, %Y')
 hits=[];low=body.lower();needle=target.lower().encode()
 for m in re.finditer(re.escape(needle),low):
  pos=m.start();fragment=body[pos:pos+5000]
  clean=' '.join(BeautifulSoup(fragment.decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split())
  title=target in clean.upper();date=expected in clean[:1400]
  header=all(x in clean[:3000] for x in ('Security','Shares','Value'))
  schedule=('SCHEDULE OF INVESTMENTS' in clean[:3000].upper())
  hits.append({'method':'inline_original_title','offset':pos,'date_near_title':date,'table_header_near_title':header,'schedule_phrase':schedule,'accepted':title and date and header and (item['form']=='N-Q' or schedule)})
 selected=next((x for x in hits if x['accepted']),None)
 toc_hits=[]
 if selected is None and item['form'] in ('N-CSR','N-CSRS'):
  for match in re.finditer(rb'<A\s+HREF="#([^"]+)"[^>]*>(.*?)</A>',body,re.I|re.S):
   label=' '.join(BeautifulSoup(match.group(2).decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split()).upper()
   if target not in label:continue
   anchor=match.group(1).decode('ascii',errors='ignore')
   anchor_match=re.search((rb'NAME="'+re.escape(anchor.encode())+rb'"'),body,re.I)
   if anchor_match is None:continue
   start=anchor_match.start();next_match=re.search(rb'NAME="toc[^"]+"',body[start+10:],re.I)
   end=(start+10+next_match.start()) if next_match else min(len(body),start+100000)
   section=body[start:end];plain=' '.join(BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser').get_text(' ',strip=True).split())
   accepted=target in plain.upper() and expected in plain and 'SCHEDULE OF INVESTMENTS' in plain.upper() and all(x in plain for x in ('Security','Shares','Value')) and 'COMMON STOCKS' in plain.upper()
   toc_hits.append({'method':'TOC_anchor','anchor':anchor,'start':start,'end':end,'section_sha256':hashlib.sha256(section).hexdigest(),'accepted':accepted})
   if accepted:selected=toc_hits[-1];break
 checks.append({'report_date':item['report_date'],'form':item['form'],'accession':item['accession'],'raw_sha256':item['raw_sha256'],
  'raw_bytes':item['raw_bytes'],'inline_hits':hits,'TOC_hits':toc_hits,'SOXX_schedule_identity_pass':selected is not None,
  'selected_identity_method':selected['method'] if selected else None,'selected_identity_location':selected})
result={'status':'eight_2013_2014_original_SOXX_schedule_identities_confirmed_source_only' if all(x['SOXX_schedule_identity_pass'] for x in checks) else '2013_2014_original_SOXX_body_identities_partial_or_unqualified',
 'plan_sha256':sha(card/'plan.json'),'checks':checks,'identity_pass_count':sum(x['SOXX_schedule_identity_pass'] for x in checks),
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'new_GETs':0,'security_values_extracted':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'full_2013_2022_coverage':False,'interpretation':'Only source identity and bounded schedule layout were checked; no original security quantities or predictive result.'}
(card/'identity_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'pass_count':result['identity_pass_count'],'methods':[x['selected_identity_method'] for x in checks]})


{'status': '2013_2014_original_SOXX_body_identities_partial_or_unqualified', 'pass_count': 2, 'methods': ['TOC_anchor', 'inline_original_title', None, None, None, None, None, None]}


## 中文结论

执行后见 `summary.json` 和 `conclusion.md`。来源资格不等于预测力或 SOXX 超额。
